# Tree capacity comparisons v005b

These CPU experiments use only approved data and the original grouped A split.
The completed combined-input tree is the control. Change the number of trees
(2,500 to 5,000) and the number of leaves (511 to 1,023) **separately**.
No validation feedback, early stopping, target cleaning or feature selection
occurs during fitting. The independent GPU sequence comparisons continue.

Default mode only reviews. Set `PA3_RUN_TREE_CAPACITY_V005B=1` and
`PA3_RESUME_TREE_CAPACITY_V005B=1` to run/reuse compatible stages. Interrupted
partial tree fits are preserved and restarted. These additional A trials are
exploratory and are excluded from the already declared v005 confirmation
inventory; a later frozen protocol is required to adopt them.


In [1]:
import os,gc,json,hashlib,time,uuid,fcntl
from pathlib import Path
from datetime import datetime,timezone
RUN=os.environ.get('PA3_RUN_TREE_CAPACITY_V005B','0')=='1'
RESUME=os.environ.get('PA3_RESUME_TREE_CAPACITY_V005B','0')=='1'
PROJECT=Path.cwd().resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():PROJECT=PROJECT/'ventilator-pressure-prediction'
OUT=PROJECT/'outputs/improvements_v005/tree_capacity'
BASE=PROJECT/'outputs/improvements_v005/tree_context'
NOTEBOOK=PROJECT/'tree_capacity_v005b.ipynb'
DATA=PROJECT/'data';SEED=20261102;THREADS=4
DTYPES=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32')
BASE_PARAMS=dict(n_estimators=2500,num_leaves=511,learning_rate=.04,min_child_samples=20,max_bin=255,reg_lambda=2.,objective='regression_l1',subsample=.9,subsample_freq=1,colsample_bytree=.9,n_jobs=4,random_state=SEED,deterministic=True,force_col_wise=True,verbosity=-1,device_type='cpu')
ARMS={'tree_combined_5000':dict(BASE_PARAMS,n_estimators=5000),'tree_combined_leaves1023':dict(BASE_PARAMS,num_leaves=1023)}
PLAN=dict(version='tree_capacity_v005b',feature_mode='combined',feature_choice='winner of completed tree_context A feature comparisons',arms=ARMS,changes='tree count and leaf count varied separately',training_breaths=61114,validation_breaths=6791,metric='native inspiratory MAE',validation_feedback=False,early_stopping=False,additional_exploratory_A_only=True,part_of_v005_capacity_confirmation_inventory=False,submission=False)
print('Tree capacity:', 'execute' if RUN else 'review', '|',OUT)


Tree capacity: execute | /home/v2gao/dsc102/pa3/gym-workspace/ventilator-pressure-prediction/outputs/improvements_v005/tree_capacity


In [2]:
if RUN:
    import hashlib
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    STEPS = 80
    V005_FEATURE_MODES = ('causal20', 'offline', 'dynamics', 'combined')
    V005_BASE_FEATURE_NAMES = (
        'R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt',
        'control_sum', 'control_mean', 'control_integral',
        'u_in_lag_1', 'u_in_change_1', 'u_in_lag_2', 'u_in_change_2',
        'u_in_lag_4', 'u_in_change_4', 'u_in_lag_8', 'u_in_change_8',
        'u_in_mean_4', 'u_in_mean_8',
    )
    V005_OFFLINE_FEATURE_NAMES = (
        'u_in_lead_1', 'u_in_lead_2', 'u_in_lead_4', 'u_in_lead_8',
        'u_in_centered_mean_4', 'u_in_centered_mean_8',
        'inspiration_duration', 'inspiration_time_fraction',
        'time_to_u_out_switch', 'inspiration_u_in_peak',
        'inspiration_u_in_peak_time', 'inspiration_u_in_mean',
        'inspiration_u_in_std', 'inspiration_u_in_integral',
    )
    V005_EWM_TIMESCALES_SECONDS = (.05, .1, .2, .5, 1., 2.)
    V005_DYNAMICS_FEATURE_NAMES = (
        'rc_tau_seconds', 'time_over_rc_tau', 'u_in_times_R', 'u_in_over_R',
        'du_in_dt', 'control_trapezoid_integral',
        'control_trapezoid_over_compliance_L',
        'control_ewm_tau_0_05', 'control_ewm_tau_0_1',
        'control_ewm_tau_0_2', 'control_ewm_tau_0_5',
        'control_ewm_tau_1', 'control_ewm_tau_2',
    )


In [3]:
if RUN:
    def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
        """Return features in original row order, after verifying 80-row blocks."""
        if len(frame) % STEPS:
            raise ValueError("Unexpected breath length; do not reshape this file")
        blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError("Breaths are not contiguous 80-row blocks")
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError("Repeated breath blocks")
        if not frame["id"].is_unique:
            raise ValueError("Duplicate row IDs")
        u = frame["u_in"].to_numpy().reshape(-1, STEPS)
        t = frame["time_step"].to_numpy().reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError("Timestamps must increase inside each breath")
        for column in ("R", "C"):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f"{column} changes within a breath")

        columns = {}
        for column in ("R", "C", "time_step", "u_in", "u_out"):
            columns[column] = frame[column].to_numpy(dtype=np.float32)
        step = np.arange(STEPS)
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        cumulative = np.cumsum(u, axis=1, dtype=np.float32)
        columns["step"] = np.broadcast_to(step, u.shape).ravel()
        columns["dt"] = dt.ravel()
        columns["control_sum"] = cumulative.ravel()
        columns["control_mean"] = (cumulative / (step + 1)).ravel()
        # Valve opening is not measured flow: this is a history proxy, not volume.
        columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
        for lag in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :lag] = u[:, :1]
            shifted[:, lag:] = u[:, :-lag]
            columns[f"u_in_lag_{lag}"] = shifted.ravel()
            columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
        padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
        for window in (4, 8):
            begin = np.maximum(0, step + 1 - window)
            columns[f"u_in_mean_{window}"] = (
                (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
            ).ravel()
        names = list(columns)
        matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
        if not np.isfinite(matrix).all() or "pressure" in names:
            raise ValueError("Invalid features")
        return matrix, names, breath_ids


In [4]:
if RUN:
    # Keep the legacy flat API available when importing these cells into a notebook.
    features_causal20 = features


    def _validate_feature_frame_v005(frame):
        """Validate known inputs only. Pressure and unrelated columns are never read."""
        if not isinstance(frame, pd.DataFrame):
            raise TypeError('Expected a pandas DataFrame')
        required = ('id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out')
        if not frame.columns.is_unique:
            raise ValueError('Duplicate column names')
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required inputs: {sorted(missing)}')
        if not len(frame) or len(frame) % STEPS:
            raise ValueError('Expected nonempty contiguous 80-row breaths')
        for column in required:
            if not pd.api.types.is_numeric_dtype(frame[column]):
                raise ValueError(f'{column} must be numeric')
            values = frame[column].to_numpy()
            if not np.isfinite(values).all():
                raise ValueError(f'{column} must be finite')
        for column in ('id', 'breath_id'):
            values = frame[column].to_numpy()
            if not np.all(values == np.floor(values)):
                raise ValueError(f'{column} must contain integer IDs')
        if not frame['id'].is_unique:
            raise ValueError('Duplicate row IDs')
        blocks = frame['breath_id'].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError('Breaths are not contiguous 80-row blocks')
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError('Repeated breath blocks')
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError('Timestamps must increase inside each breath')
        for column, levels in (('R', (5, 20, 50)), ('C', (10, 20, 50))):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f'{column} changes within a breath')
            if not np.isin(values, levels).all():
                raise ValueError(f'Unexpected {column} level')
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        if not np.isin(out, (0, 1)).all():
            raise ValueError('u_out must be binary')
        # The approved data has a single inspiration followed by expiration. A fully
        # observed inspiration without an expiration sample is supported as well.
        if np.any(out[:, 0] != 0) or np.any(np.diff(out.astype(np.int8), axis=1) < 0):
            raise ValueError('Expected initial inspiration followed by at most one valve opening')
        return breath_ids


    def feature_names_v005(feature_mode='causal20'):
        if feature_mode not in V005_FEATURE_MODES:
            raise ValueError(f'Unknown feature_mode: {feature_mode!r}')
        names = list(V005_BASE_FEATURE_NAMES)
        if feature_mode in ('offline', 'combined'):
            names.extend(V005_OFFLINE_FEATURE_NAMES)
        if feature_mode in ('dynamics', 'combined'):
            names.extend(V005_DYNAMICS_FEATURE_NAMES)
        return names


    def _offline_feature_block_v005(frame):
        """Known whole-breath input context; no target values or statistics."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        inspiration = out == 0
        n_breaths = len(u)
        columns = []
        for lead in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :-lead] = u[:, lead:]
            shifted[:, -lead:] = u[:, -1:]
            columns.append(shifted)
        prefix = np.concatenate((np.zeros((n_breaths, 1)), np.cumsum(u, axis=1)), axis=1)
        steps = np.arange(STEPS)
        for window in (4, 8):
            # Even centered windows have window//2 prior and window//2-1 future
            # neighbours, matching pandas rolling(center=True, min_periods=1).
            begin = np.maximum(0, steps - window//2)
            end = np.minimum(STEPS, steps + (window+1)//2)
            columns.append((prefix[:, end] - prefix[:, begin]) / (end-begin))
        has_expiration = np.any(out == 1, axis=1)
        first_expiration = np.argmax(out == 1, axis=1)
        switch_index = np.where(has_expiration, first_expiration, STEPS-1)
        switch_time = t[np.arange(n_breaths), switch_index]
        duration = switch_time - t[:, 0]
        # With strictly increasing 80-step timestamps and initial inspiration,
        # duration is positive, including the no-expiration boundary case.
        fraction = np.clip((t-t[:, :1])/duration[:, None], 0., 1.)
        until_switch = np.maximum(switch_time[:, None]-t, 0.)
        masked = np.where(inspiration, u, -np.inf)
        peak_index = np.argmax(masked, axis=1)  # first peak resolves a known-input tie
        peak = u[np.arange(n_breaths), peak_index]
        peak_time = t[np.arange(n_breaths), peak_index] - t[:, 0]
        count = inspiration.sum(axis=1)
        mean = np.sum(np.where(inspiration, u, 0.), axis=1)/count
        variance = np.sum(np.where(inspiration, (u-mean[:, None])**2, 0.), axis=1)/count
        std = np.sqrt(np.maximum(variance, 0.))
        # Integrate between inspiratory observations only; do not infer unobserved
        # flow or include the interval crossing into expiration.
        both_inspiration = inspiration[:, 1:] & inspiration[:, :-1]
        integral = np.sum(np.where(both_inspiration,
            .5*(u[:, 1:]+u[:, :-1])*np.diff(t, axis=1), 0.), axis=1)
        def broadcast(values):
            return np.broadcast_to(values[:, None], u.shape)
        columns.extend((broadcast(duration), fraction, until_switch, broadcast(peak),
            broadcast(peak_time), broadcast(mean), broadcast(std), broadcast(integral)))
        return np.stack(columns, axis=-1).astype(np.float32)


    def _dynamics_feature_block_v005(frame):
        """Causal timing and control proxies; valve opening is not measured flow."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        r = frame['R'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        c_litres = frame['C'].to_numpy(dtype=np.float64).reshape(-1, STEPS)/1000.
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        tau = r*c_litres  # R in cmH2O/(L/s), C converted from mL/cmH2O
        derivative = np.zeros_like(u)
        derivative[:, 1:] = np.diff(u, axis=1)/dt[:, 1:]
        trapezoid = np.zeros_like(u)
        trapezoid[:, 1:] = np.cumsum(.5*(u[:, 1:]+u[:, :-1])*dt[:, 1:], axis=1)
        # These interactions can help the learner represent R/C sensitivity. They
        # are not asserted to have physical pressure/flow/volume units because u_in
        # is a valve-opening percentage, not observed flow or driving pressure.
        columns = [tau, (t-t[:, :1])/tau, u*r, u/r, derivative,
                   trapezoid, trapezoid/c_litres]
        scales = np.asarray(V005_EWM_TIMESCALES_SECONDS, dtype=np.float64)
        histories = np.empty((*u.shape, len(scales)), dtype=np.float32)
        state = np.broadcast_to(u[:, :1], (len(u), len(scales))).copy()
        histories[:, 0, :] = state
        # One loop over 80 timesteps; all breaths and timescales run together.
        # Irregular-time EWMA includes the control observed at the current step.
        for step in range(1, STEPS):
            alpha = np.exp(-dt[:, step, None]/scales[None, :])
            state = alpha*state + (1.-alpha)*u[:, step, None]
            histories[:, step, :] = state
        fixed = np.stack(columns, axis=-1).astype(np.float32)
        return np.concatenate((fixed, histories), axis=-1)


    def make_sequence_features(frame, feature_mode='causal20'):
        """Return (X[breaths,80,F] float32, names, breath_ids), in input order.

        causal20: exact legacy features; offline: legacy + known future/context;
        dynamics: legacy + causal dynamics proxies; combined: all three blocks.
        No scaling, fitting, target reads, IDs as features, or frame mutation occur.
        """
        names = feature_names_v005(feature_mode)
        breath_ids = _validate_feature_frame_v005(frame)
        base_flat, base_names, old_ids = features_causal20(frame)
        if base_names != list(V005_BASE_FEATURE_NAMES) or not np.array_equal(breath_ids, old_ids):
            raise ValueError('Legacy feature schema changed')
        base = base_flat.reshape(-1, STEPS, len(base_names))
        if feature_mode == 'causal20':
            return base, names, breath_ids.copy()
        result = np.empty((len(breath_ids), STEPS, len(names)), dtype=np.float32)
        result[..., :len(base_names)] = base
        position = len(base_names)
        if feature_mode in ('offline', 'combined'):
            block = _offline_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if feature_mode in ('dynamics', 'combined'):
            block = _dynamics_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if position != len(names) or not np.isfinite(result).all():
            raise ValueError('Invalid feature output')
        return result, names, breath_ids.copy()


In [5]:
def canonical_hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()
def digest(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
def array_hash(value):return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
def code_hash(path):
 n=json.loads(Path(path).read_text());return hashlib.sha256('\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code').encode()).hexdigest()
def write_json(path,value):
 temp=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex);temp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');os.replace(temp,path)
def save_array(path,value):
 temp=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
 with temp.open('wb') as f:np.save(f,value,allow_pickle=False)
 os.replace(temp,path)
def aligned_score(prediction,target,ids,expected):
 p=np.asarray(prediction,dtype=np.float64);y=np.asarray(target,dtype=np.float64)
 assert p.ndim==1 and p.shape==y.shape and np.isfinite(p).all() and np.isfinite(y).all()
 assert np.array_equal(ids,expected)
 return float(np.abs(p-y).mean())
def event(kind,**values):
 with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),event=kind,**values),allow_nan=False)+'\n')


In [6]:
if RUN:
 import lightgbm as lgb
 OUT.mkdir(parents=True,exist_ok=True)
 WORKER_LOCK=(OUT/'execution-worker.lock').open('a+');fcntl.flock(WORKER_LOCK,fcntl.LOCK_EX|fcntl.LOCK_NB)
 quality=json.loads((PROJECT/'quality_audit_v002.json').read_text());train_sha=digest(DATA/'train.csv')
 assert train_sha==quality['files']['train.csv']['sha256']=='2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7'
 base_design=json.loads((BASE/'design.json').read_text());assert base_design['source_sha256']==code_hash(PROJECT/'tree_context_v005.ipynb')
 controls=[json.loads(p.read_text()) for p in BASE.glob('record_A_tree_*.json')];assert len(controls)==4 and all(d['status']=='complete' for d in controls)
 winner=min(controls,key=lambda d:d['validation_mae']);assert winner['name']=='tree_combined'
 control=json.loads((BASE/'record_A_tree_combined.json').read_text());assert control['recipe']['fit_params']==BASE_PARAMS
 assert control['design_sha256']==canonical_hash(base_design) and control['source_sha256']==base_design['source_sha256']
 assert digest(control['checkpoint_path'])==control['checkpoint_sha256'] and digest(control['prediction_file'])==control['prediction_sha256']
 assert digest(control['scored_row_id_file'])==control['scored_row_id_sha256']
 frame=pd.read_csv(DATA/'train.csv',dtype=DTYPES);BREATH_IDS=_validate_feature_frame_v005(frame);assert len(BREATH_IDS)==67905
 Y=frame.pressure.to_numpy(copy=True).reshape(-1,80);UOUT=frame.u_out.to_numpy(copy=True).reshape(-1,80);ROW=frame.id.to_numpy(copy=True).reshape(-1,80);RC=frame[['R','C']].to_numpy(copy=True)[::80]
 KNOWN_FRAME=frame[['id','breath_id','R','C','time_step','u_in','u_out']].copy();del frame;gc.collect()
 with np.load(BASE/'split_breath_ids.npz',allow_pickle=False) as saved:
  GROUP_IDS={k:saved[k] for k in ['A_fit','A_validation']}
 index=pd.Index(BREATH_IDS);GROUPS={k:np.sort(index.get_indexer(v)) for k,v in GROUP_IDS.items()}
 assert len(GROUPS['A_fit'])==61114 and len(GROUPS['A_validation'])==6791
 assert all((v>=0).all() and len(np.unique(v))==len(v) for v in GROUPS.values())
 assert np.array_equal(np.sort(np.r_[GROUPS['A_fit'],GROUPS['A_validation']]),np.arange(67905))
 GROUP_HASHES={k:array_hash(BREATH_IDS[v]) for k,v in GROUPS.items()};assert GROUP_HASHES==base_design['group_hashes']
 AI=GROUPS['A_validation'];AMASK=UOUT[AI]==0;TARGET=Y[AI][AMASK].astype(np.float64);SCORED_IDS=ROW[AI][AMASK];assert len(TARGET)==206098
 assert np.array_equal(SCORED_IDS,np.load(BASE/'A_validation_row_ids.npy'))
 assert abs(aligned_score(np.load(control['prediction_file']),TARGET,np.load(control['scored_row_id_file']),SCORED_IDS)-control['validation_mae'])<1e-10
 DESIGN=dict(plan=PLAN,source_sha256=code_hash(NOTEBOOK),train_sha256=train_sha,group_hashes=GROUP_HASHES,feature_names=feature_names_v005('combined'),baseline_design_sha256=canonical_hash(base_design),baseline_record_sha256=digest(BASE/'record_A_tree_combined.json'),baseline_A_mae=control['validation_mae'])
 if (OUT/'design.json').exists():assert RESUME and json.loads((OUT/'design.json').read_text())==DESIGN
 else:
  save_array(OUT/'A_validation_row_ids.npy',SCORED_IDS)
  with (OUT/'split_breath_ids.npz').open('wb') as f:np.savez_compressed(f,**GROUP_IDS)
  write_json(OUT/'design.json',DESIGN)
 assert np.array_equal(np.load(OUT/'A_validation_row_ids.npy'),SCORED_IDS)
 event('initialized',source_sha256=DESIGN['source_sha256'],resume=RESUME,lightgbm=lgb.__version__)
 FEATURE_CACHE={}


In [7]:
def scored_x(group):
 if group not in FEATURE_CACHE:
  indices=GROUPS[group];positions=(indices[:,None]*80+np.arange(80)[None,:]).ravel()
  raw,names,ids=make_sequence_features(KNOWN_FRAME.iloc[positions],'combined')
  assert names==DESIGN['feature_names'] and np.array_equal(ids,BREATH_IDS[indices])
  FEATURE_CACHE[group]=np.ascontiguousarray(raw[UOUT[indices]==0],dtype=np.float32)
 return FEATURE_CACHE[group]

def fit_arm(name):
 params=ARMS[name];trees=params['n_estimators'];path=OUT/('A_'+name+'.txt');predpath=OUT/('A_'+name+'_predictions.npy');recordpath=OUT/('record_A_'+name+'.json');progress=OUT/('A_'+name+'_progress.jsonl')
 identity=dict(design_sha256=canonical_hash(DESIGN),name=name,fit_breath_sha256=GROUP_HASHES['A_fit'],validation_breath_sha256=GROUP_HASHES['A_validation'],feature_mode='combined',input_columns=DESIGN['feature_names'],fit_params=params)
 if recordpath.exists():
  assert RESUME
  r=json.loads(recordpath.read_text());assert r['status']=='complete' and r['identity']==identity
  assert digest(path)==r['checkpoint_sha256'] and digest(predpath)==r['prediction_sha256']
  assert abs(aligned_score(np.load(predpath),TARGET,np.load(OUT/'A_validation_row_ids.npy'),SCORED_IDS)-r['validation_mae'])<1e-10
  booster=lgb.Booster(model_file=str(path));assert booster.num_trees()==trees and booster.feature_name()==DESIGN['feature_names'];del booster
  return r
 partial=[p for p in [path,predpath,progress,path.with_name(path.name+'.tmp.txt')] if p.exists()]
 if partial:
  archive=OUT/'archive'/('interrupted_'+name+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'));archive.mkdir(parents=True)
  for p in partial:p.rename(archive/p.name)
  event('interrupted_artifacts_preserved',member=name,directory=str(archive))
 start=time.monotonic();event('trial_started',name=name,fit_params=params)
 learner=lgb.LGBMRegressor(**params)
 def callback(env):
  if (env.iteration+1)%250==0:
   with progress.open('a') as f:f.write(json.dumps(dict(event='trees',iteration=env.iteration+1,elapsed_seconds=time.monotonic()-start,validation_feedback=False))+'\n')
 callback.order=10;callback.before_iteration=False
 ix=GROUPS['A_fit'];learner.fit(scored_x('A_fit'),Y[ix][UOUT[ix]==0],feature_name=DESIGN['feature_names'],callbacks=[callback]);fit_seconds=time.monotonic()-start
 assert learner.booster_.num_trees()==trees and learner.booster_.feature_name()==DESIGN['feature_names']
 temporary=path.with_name(path.name+'.tmp.txt');learner.booster_.save_model(str(temporary),num_iteration=trees);os.replace(temporary,path)
 prediction=np.asarray(learner.predict(scored_x('A_validation'),num_iteration=trees,num_threads=4),dtype=np.float64);mae=aligned_score(prediction,TARGET,SCORED_IDS,SCORED_IDS);save_array(predpath,prediction)
 diag_ix=GROUPS['A_fit'][:2048];diag_mask=UOUT[diag_ix]==0;count=int(diag_mask.sum());fit_pred=learner.predict(scored_x('A_fit')[:count],num_iteration=trees,num_threads=4);fit_mae=float(np.abs(fit_pred-Y[diag_ix][diag_mask]).mean())
 errors=np.abs(prediction-TARGET);pairs=np.repeat(RC[AI],80,axis=0)[AMASK.ravel()];table=pd.DataFrame(dict(R=pairs[:,0],C=pairs[:,1],absolute_error=errors)).groupby(['R','C']).absolute_error.agg(['size','mean','sum']).reset_index();table['total_error_share']=table['sum']/errors.sum();rcpath=OUT/('A_'+name+'_by_rc.csv');table.to_csv(rcpath,index=False)
 r=dict(record_type='trial',status='complete',name=name,split='A',kind='fresh_tree',feature_mode='combined',n_features=len(DESIGN['feature_names']),input_columns=DESIGN['feature_names'],seed=SEED,recipe=dict(name=name,kind='fresh_tree',feature_mode='combined',seed=SEED,params=params,fit_params=params,fixed_trees=trees,rc='numeric'),identity=identity,identity_sha256=canonical_hash(identity),design_sha256=canonical_hash(DESIGN),source_sha256=DESIGN['source_sha256'],training_breaths=61114,validation_breaths=6791,scored_points=len(TARGET),validation_mae=mae,training_diagnostic_mae=fit_mae,training_diagnostic_breaths=2048,fit_seconds=fit_seconds,runtime_seconds=time.monotonic()-start,reused_checkpoint=False,best_iteration=trees,best_epoch=None,validation_feedback=False,checkpoint_path=str(path),checkpoint_sha256=digest(path),prediction_file=str(predpath),prediction_sha256=digest(predpath),scored_row_id_file=str(OUT/'A_validation_row_ids.npy'),scored_row_id_sha256=digest(OUT/'A_validation_row_ids.npy'),rc_breakdown_file=str(rcpath),rc_breakdown_sha256=digest(rcpath),progress_file=str(progress),termination='fixed_tree_budget',eligible_for_current_confirmation_round=False)
 write_json(recordpath,r);event('trial_completed',name=name,validation_mae=mae,training_diagnostic_mae=fit_mae,runtime_seconds=r['runtime_seconds']);print(name,mae,flush=True)
 del learner;gc.collect();return r


In [8]:
if RUN:
    fit_arm('tree_combined_5000')

tree_combined_5000 0.48439081052016447


In [9]:
if RUN:
    fit_arm('tree_combined_leaves1023')

tree_combined_leaves1023 0.4813969457152611


In [10]:
if RUN:
    event('comparisons_complete',new_fits=2,C_read=False,submission=False)
    fcntl.flock(WORKER_LOCK,fcntl.LOCK_UN);WORKER_LOCK.close()
for p in sorted(OUT.glob('record_A_*.json')) if OUT.exists() else []:
    d=json.loads(p.read_text());print(d['name'],d['validation_mae'],d['runtime_seconds'])


tree_combined_5000 0.48439081052016447 954.7913220822811
tree_combined_leaves1023 0.4813969457152611 637.9338708203286
